# 09 · v1 / v2 training workflow

Train champion (older window) and challenger (fresher window) with the same LightGBM recipe. Both later score the same Aug 5–21 holdout.

In [ ]:
import pandas as pd

from cross_model_drift.data import load_split
from cross_model_drift.features import target_vector
from cross_model_drift.metrics import quality_metrics
from cross_model_drift.models import load_model, train_lightgbm
from cross_model_drift.notebook import setup_model_session
from cross_model_drift.tracking import clearml_task, log_metrics

nb = setup_model_session()
tuned_path = nb.artifacts / "models" / "v1_lightgbm_tuned.joblib"
extra_params = {}
if tuned_path.exists():
    extra_params = load_model(tuned_path).params
extra_params

In [ ]:
rows = []
for version in ("v1", "v2"):
    train = load_split(version, "train", nb.config, engine=nb.engine)
    valid = load_split(version, "validation", nb.config, engine=nb.engine)
    test = load_split(version, "test", nb.config, engine=nb.engine)
    model = train_lightgbm(
        train,
        target_vector(train),
        valid,
        target_vector(valid),
        params=extra_params,
        threshold=nb.threshold,
    )
    metrics = quality_metrics(target_vector(test), model.predict_proba(test), threshold=nb.threshold)
    metrics["version"] = version
    metrics["n_train"] = len(train)
    path = model.save(nb.artifacts / "models" / f"{version}_champion_challenger.joblib")
    rows.append(metrics)
    with clearml_task(f"train_{version}", config=nb.config, task_type="training", tags=[version], init=True) as task:
        log_metrics(task, metrics, title=f"{version}_test")
    print(version, path)
pd.DataFrame(rows).set_index("version")